# Variance Gamma: a pure-jump model via a random clock

Every model so far has had a Brownian diffusion at its core: BSM is pure
diffusion, Heston is diffusion with stochastic variance, Merton/Kou/Bates are
diffusion plus jumps. Variance Gamma (Madan, Carr, Chang 1998) throws the
diffusion away entirely. It is a **pure-jump** process, no continuous Brownian
part at all.

Yet its paths look almost continuous. VG is an **infinite-activity** jump
process: infinitely many jumps in any interval, but almost all infinitesimally
small, with only a few large. Contrast the finite-activity jump-diffusions
(Merton, Kou), which have a finite number of meaningful jumps per year. VG is
"a dense spray of mostly-tiny jumps, a few large," and it captures skew and fat
tails through the distribution of jump sizes rather than a diffusion-plus-jump
split.

VG is the contrast piece in the cross-model benchmark: three parameters, no
diffusion, versus Bates's eight with both diffusion and jumps. The sharp
question is whether a parsimonious pure-jump model fits the SPX smile
competitively with the rich Bates model.

## 1. Subordination: Brownian motion under a random clock

VG is built by running a drifting Brownian motion on a **random time clock**.
Evaluate the Brownian motion not at calendar time $t$ but at a random, increasing
amount of *business time* $\Gamma_t$:

$$X_t = \theta\,\Gamma_t + \sigma\,W_{\Gamma_t}$$

where $\Gamma_t$ is a **Gamma process** (the random clock) and $W$ is a standard
Brownian motion.

The direction matters and is easy to get backwards. Calendar time $t$ is *not*
random, it is the ordinary wall clock, the deterministic input. What is random is
the *business time* $\Gamma_t$, the amount of market activity accumulated by
calendar time $t$. At each fixed $t$, $\Gamma_t$ is Gamma-distributed with mean
$t$ and variance $\nu t$: business time tracks calendar time on average but
wobbles around it. As $t$ advances, $\Gamma_t$ advances by random increments,
sometimes racing ahead (a chaotic news period), sometimes crawling (a quiet one).

**Why "subordination".** When one process $W$ is evaluated at a random time given
by an increasing process $\Gamma$, the process $\Gamma$ is the *subordinator* and
$W_{\Gamma_t}$ is *subordinated* to it (Bochner, 1940s). The word names a
hierarchy: the clock $\Gamma$ governs the time at which $W$ is observed, so $W$ is
*subordinate to* the clock, it does not run on its own time, it runs on whatever
time the clock dictates. The clock must be non-decreasing (time only moves
forward); the Gamma process qualifies, being increasing with independent
Gamma-distributed increments.

The economic story: markets do not experience time uniformly. Some periods pack
in lots of information and trading (fast business time), others are quiet (slow).
Over one calendar hour, sometimes ten "business hours" of activity happen,
sometimes ten minutes' worth. A return is a Brownian increment over the *business*
time elapsed, so a chaotic hour produces a big move and a dead hour a tiny one.
The random clock turns uniform-variance Brownian increments into the fat-tailed,
variable-magnitude returns markets actually show.

## 2. The three parameters

- $\sigma$: volatility of the underlying Brownian motion. Overall scale.
- $\nu$: variance rate of the Gamma clock. Controls **kurtosis** (fat tails).
  $\nu \to 0$ makes the clock deterministic (business time = calendar time) and
  recovers pure Brownian motion; larger $\nu$ means a more erratic clock and
  fatter tails.
- $\theta$: drift of the Brownian motion in business time. Controls **skew**.
  $\theta < 0$ (down-drift) gives the left-skew of equity returns; $\theta = 0$
  gives a symmetric distribution.

A clean decomposition: $\sigma$ scales, $\nu$ fattens, $\theta$ tilts.

## 3. The distribution as a mixture of Gaussians

Conditional on the clock taking a specific value $\Gamma_t = g$, the return is
just Brownian motion observed at time $g$:

$$X_t \mid (\Gamma_t = g) \;\sim\; \mathcal{N}(\theta g,\ \sigma^2 g)$$

So each possible clock value gives a *different* Gaussian: a quiet-clock draw
(small $g$) is narrow and near zero; a busy-clock draw (large $g$) is wide *and*
more shifted (the drift $\theta$ accumulates over more business time). The actual
return distribution is the probability-weighted blend of all of them:

$$f_{X_t}(x) = \int_0^\infty \underbrace{f_{\mathcal{N}(\theta g,\ \sigma^2 g)}(x)}_{\text{conditional Gaussian}} \cdot \underbrace{f_\Gamma(g)}_{\text{clock density}}\ dg$$

a **continuous mixture of Gaussians**, one per clock value, weighted by the Gamma
density. Mechanically (and this is how we simulate it): draw a clock value $g$
from the Gamma, draw a return from $\mathcal{N}(\theta g, \sigma^2 g)$, repeat.
The pooled histogram is fatter-tailed than any single Gaussian, because the
wide-clock draws over-populate the tails. **That variance-mixing is where VG's
excess kurtosis comes from**, and the down-drift making wide draws also more
negative is where the skew comes from.

## 4. The characteristic function

Subordination pays off: compute the char func by conditioning on the clock, then
averaging. Conditional on $\Gamma_t = g$, the char func is the Gaussian's,
$\exp(g\,[iu\theta - \tfrac12\sigma^2 u^2])$, of the form $\exp(g \cdot s)$. Then
average over the Gamma-distributed $g$:

$$\phi(u) = \mathbb{E}_g\big[e^{g\,s}\big], \qquad s = iu\theta - \tfrac12\sigma^2 u^2.$$

But $\mathbb{E}_g[e^{gs}]$ is the **moment generating function of the Gamma
distribution**, which is a *power law*: $(1 - \nu s)^{-t/\nu}$. Hence

$$\phi_{VG}(u) = \left(1 - iu\theta\nu + \tfrac12\sigma^2\nu u^2\right)^{-t/\nu}.$$

The power-law shape comes directly from the Gamma clock's MGF being a power law.
This is the deep tidy fact: a model's char func inherits the functional form of
the MGF of its driving randomness. The jump-diffusions have
exponential-of-exponential char funcs because their randomness is Poisson; VG has
a power-law char func because its randomness is a Gamma clock. It is arguably the
simplest char func of all the models here, a nice payoff for the conceptual
depth.

For the log-*price*, prepend the drift and a martingale correction:

$$\phi_{\ln S_T}(u) = e^{iu(\ln S_0 + (r - q + \omega)t)}\,\phi_{VG}(u),
\qquad \omega = \frac{1}{\nu}\log\!\left(1 - \theta\nu - \tfrac12\sigma^2\nu\right).$$

$\omega$ is the convexity correction (analogue of the jump compensator
$-\lambda\kappa_J$) that forces $\mathbb{E}[S_T] = S_0 e^{(r-q)t}$. It hides a
**parameter constraint**: the log's argument $1 - \theta\nu - \tfrac12\sigma^2\nu$
must be positive, or $\omega$ diverges. This bounds $\theta, \nu, \sigma$
together, the VG analogue of Kou's $\eta_1 > 1$ ("the process has finite mean").

## 5. Where VG sits, and what the benchmark should show

VG captures skew and kurtosis with three parameters and no diffusion, against
Bates's eight. It is known to fit a *single* maturity's smile well but to
struggle with the *term structure*: with no stochastic volatility, its skew and
kurtosis decay with maturity in a fixed way (roughly, skew $\sim \theta$ and
kurtosis $\sim \nu/t$, both shrinking as $t$ grows) that the market does not
follow.

So the expected benchmark finding is the mirror image of Heston. Heston had a
term structure but too little short-dated skew; VG has plenty of single-slice
skew/kurtosis but the wrong term structure. Bates, with both stochastic vol and
jumps, should be the one that gets both. That contrast, parsimonious-but-rigid VG
versus rich-but-heavy Bates, is exactly why VG earns its place in the benchmark.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# from models.vg import vg_char_func, vg_cumulants, vg_simulate_terminal
# from pricing.fourier import vg_cos_price
# from models.bsm import bsm_price   # for the nu -> 0 (Brownian) limit check

In [ ]:
# DEMO 1 (the centerpiece): mixture-of-Gaussians, kurtosis from variance-mixing.
#   - draw many clock values g ~ Gamma(mean=t, var=nu*t)
#   - overlay a few conditional Gaussians N(theta*g, sigma^2*g) for small/mid/large g
#     (narrow-near-zero vs wide-and-shifted), thin lines
#   - then the pooled VG return histogram (draw g, draw N(theta g, sigma^2 g)),
#     with a matched-variance single Gaussian overlaid
#   - teaching point: VG histogram is peakier AND fatter-tailed than the matched
#     Gaussian. Peak + fat tails = excess kurtosis, born from mixing variances.
#     With theta<0 it also leans left = skew

# DEMO 2: parameter effects. Sweep nu (kurtosis) and theta (skew) separately,
#   show the return density changing. nu -> 0 collapses toward the Gaussian.

# DEMO 3: VG smile vs maturity (the term-structure weakness). Price VG smiles at
#   several T, invert to IV. Show skew/kurtosis decaying with T in VG's fixed way.
#   Sets up the benchmark contrast with Bates (which controls term structure via
#   stochastic vol).

In [ ]:
# Validation, ground-truth-first, mirroring the jump models:
#   Gate 1: nu -> 0 recovers Black-Scholes (clock deterministic -> pure BM).
#           vg_cos_price at small nu vs bsm_price at matched sigma.
#   Gate 2: martingale phi(-i) = S0 exp((r-q)t), certifies omega.
#   Gate 3: phi(0) = 1.
#   Gate 4: parameter constraint 1 - theta*nu - 0.5*sigma^2*nu > 0 enforced/raised.
#   Gate 5: cumulants reduce sensibly; c1, c2 checked against known VG moments.
#   Gate 6: COS vs Monte Carlo (draw Gamma clock, then Gaussian; z-score < 3).
#           The independent ground truth for the char-func pricer.